Plant Growth (Biomass)

Plants grow logistically based on current biomass and nutrient availability.$$\frac{dB}{dt} = \mu \cdot B \cdot \left(1 - \frac{B}{K}\right) \cdot \frac{C}{C + K_c} \cdot f(H)$$

$\mu$: Maximum growth rate.

$K$: Carrying capacity (max size of the plant).$\frac{C}{C + K_c}$: Michaelis-Menten kinetics (growth slows if nutrients $C$ are low).

$f(H)$: A "bell curve" function that slows growth if pH is outside the 5.5–6.5 range

Nutrient Concentration (EC)Nutrients decrease as the plant eats them and increase when the RL agent adds "Dose."$$\frac{dC}{dt} = \frac{1}{V} \left( \text{Dose}(t) - \alpha \cdot \frac{dB}{dt} \right) + \sigma_c \frac{dW}{dt}$$

$V$: Reservoir volume.

$\alpha$: Nutrient-to-biomass conversion ratio (how much "food" creates 1g of plant).

$\sigma_c \frac{dW}{dt}$: This is the Stochastic (SDE) part from the paper—it represents random sensor noise or evaporation spikes.

pH DynamicspH is highly sensitive. It changes based on nutrient uptake (plants release ions) and agent "Adjusters."$$\frac{dH}{dt} = \beta \cdot \frac{dB}{dt} + \gamma \cdot \text{pH\_Adjuster}(t) + \sigma_h \frac{dW}{dt}$$

$\beta$: Biological acidification constant.

$\gamma$: Potency of your pH-up/down solution.

To make your Mechanistic Model work for RL training without historical data, we need to categorize these variables into three groups: Drivers (Environment), System States (Water/Nutrients), and Outcomes (Plant growth).

Here is how these variables should interact in your ODE/SDE system, following the "coupled dynamics" logic from the paper:

Environmental Drivers (The "Forcing" Functions)
These are your external stochastic variables. In the paper's framework, these would be the noise-driven components.

Temperature & Humidity: These two define the Vapor Pressure Deficit (VPD).Mechanism: High Temp + Low Humidity = High VPD $\rightarrow$ Massive water uptake, potential wilting.Light Intensity: This is the "energy" input.Mechanism: Without light, the photosynthesis ODE equals zero, and nutrient uptake stops.

Physical & Chemical States (The "Controlled" Variables)
This is where your RL agent will spend most of its effort.

pH Level: Highly volatile. You should model this with a "drift" (as plants excrete ions) and "diffusion" (random chemical fluctuations).

TDS (Total Dissolved Solids): This replaces the "EC" in our previous model. It represents the concentration of nutrients.

Crucial Logic: TDS must be coupled with Water Volume. If water evaporates, TDS rises even if the amount of salts stays the same.

Biological Outcomes (The "Latent" Growth Model)
These are the variables that tell the RL agent if it is doing a good job (the Reward signals).

Greenness (e.g., SPAD/Chlorophyll): This is a proxy for nitrogen health.

Mechanism: If TDS is too low for too long, Greenness drops.

Leaf Area & Plant Health: These are your "integrators." They don't change instantly; they represent the sum of all past environment states.

Mechanism: If pH stays outside the 5.5–6.5 range for 24 hours, "Plant Health" begins an exponential decay